In [1]:
import pandas as pd
import numpy as np

In [2]:
book = pd.read_csv("Books.csv")
user = pd.read_csv("Users.csv")
rating = pd.read_csv("Ratings.csv")

C:\Users\prash\AppData\Local\Temp\ipykernel_17524\2150940534.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  book = pd.read_csv("Books.csv")


In [3]:
book.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [4]:
user.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [5]:
rating.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [6]:
book.isna().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [7]:
user.isna().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [8]:
rating.isna().sum()

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

# Popularity-based System

In [10]:
book_rating = book.merge(rating, on="ISBN")

In [11]:
num_ratings = book_rating.groupby("Book-Title").count()["Book-Rating"].reset_index()
num_ratings.rename(columns={"Book-Rating": "Num-Rating"}, inplace=True)

avg_ratings = book_rating.groupby("Book-Title")["Book-Rating"].mean().reset_index()
avg_ratings.rename(columns={"Book-Rating": "Avg-Rating"}, inplace=True)

In [12]:
popular_df = avg_ratings.merge(num_ratings, on="Book-Title")

In [13]:
popular_df = popular_df[popular_df["Num-Rating"] > 250].sort_values("Avg-Rating", ascending=False).head(50)

In [14]:
popular_df = popular_df.merge(book, on="Book-Title").drop_duplicates("Book-Title")[["Book-Title", "Book-Author", "Num-Rating", "Avg-Rating", "Image-URL-S"]]

In [15]:
popular_df.head()

,Book-Title,Book-Author,Num-Rating,Avg-Rating,Image-URL-S
0,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,428,5.852804,http://images.amazon.com/images/P/0439136350.0...
3,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,387,5.824289,http://images.amazon.com/images/P/0439139597.0...
5,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,278,5.737410,http://images.amazon.com/images/P/0590353403.0...
9,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,347,5.501441,http://images.amazon.com/images/P/043935806X.0...
13,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,556,5.183453,http://images.amazon.com/images/P/0439064872.0...


# Collaborative Filtering

In [17]:
temp = book_rating.groupby("User-ID").count()["Book-Rating"] > 200
good_users = temp[temp].index

In [18]:
filtered_ratings = book_rating[book_rating["User-ID"].isin(good_users)]

In [19]:
temp = filtered_ratings.groupby("Book-Title").count()["Book-Rating"] >= 50
good_books = temp[temp].index

In [20]:
final_df = filtered_ratings[filtered_ratings["Book-Title"].isin(good_books)]

In [21]:
final_df = final_df.pivot_table(index="Book-Title", columns="User-ID", values="Book-Rating")

In [22]:
pt = final_df.fillna(0)
pt

User-ID,254,2276,2766,2977,3363,4017,4385,6251,6323,6543,...,271705,273979,274004,274061,274301,274308,275970,277427,277639,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Bend in the Road,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(pt)

In [24]:
def recommend(book):
    index = np.where(pt.index == book)[0][0]
    distance = similarity_scores[index]
    rec_book = sorted(list(enumerate(distance)), reverse=True, key=lambda x:x[1])[1:6]

    for i in rec_book:
        print(pt.index[i[0]])

In [25]:
recommend("1984")

Animal Farm
The Handmaid's Tale
Brave New World
The Vampire Lestat (Vampire Chronicles, Book II)
The Hours : A Novel


In [26]:
import joblib

joblib.dump(pt, "pivot_table.joblib")
joblib.dump(popular_df, "popularity.joblib")
joblib.dump(similarity_scores, "similarity.joblib")

['similarity.joblib']